In [ ]:
import hashlib
import itertools
from pathlib import Path

import numpy as np
import pandas as pd

In [ ]:
SPECIES = "SalamanderID2025"
EXPECTED_SHA256 = "38f777b8309fc9709b933cef0f55b8f135efd9960bc973369a8c945bd8739356"
TRANSITIONS = [
    (5616, 5617),
    (5640, 5644),
    (5642, 5643),
    (5538, 5539),
    (5613, 5614),
    (5375, 5804),
    (5741, 5830),
    (5667, 5873),
    (5703, 5729),
    (5774, 5960),
]

In [ ]:
def first_existing(paths):
    for path in paths:
        path = Path(path)
        if path.exists():
            return path
    raise FileNotFoundError([str(path) for path in paths])

work_dir = Path.cwd()
input_root = Path("/kaggle/input")

data_roots = [
    Path("/kaggle/input/competitions/animal-clef-2026"),
    Path("/kaggle/input/animal-clef-2026"),
    work_dir / "animal-clef-2026",
    work_dir / "data" / "animal-clef-2026",
    work_dir / "01_DATASET_AND_REFERENCES" / "animal-clef-2026",
    work_dir.parent / "AnimalCLEF2026" / "01_DATASET_AND_REFERENCES" / "animal-clef-2026",
]
if input_root.exists():
    data_roots.extend(sorted(path for path in input_root.glob("*") if (path / "metadata.csv").exists() and (path / "sample_submission.csv").exists()))

data_root = first_existing(path for path in data_roots if (Path(path) / "metadata.csv").exists() and (Path(path) / "sample_submission.csv").exists())

base_submissions = [
    work_dir / "input" / "submission_032903_base.csv",
    work_dir / "submission_032903_base.csv",
    work_dir.parent / "input" / "submission_032903_base.csv",
    Path("/kaggle/input/animalclef2026-lampr/input/submission_032903_base.csv"),
    Path("/kaggle/input/animalclef2026-lampr/submission_032903_base.csv"),
    Path("/kaggle/input/lb-0-32903-lynx-aliked-merge-only-reproducer/animalclef_lb032903_lynx_aliked_reproducer_v20260430/submission.csv"),
    Path("/kaggle/input/notebooks/hanifnoerrofiq/lb-0-32903-lynx-aliked-merge-only-reproducer/animalclef_lb032903_lynx_aliked_reproducer_v20260430/submissions/submission_032684_lynx_aliked_prob098_mergeonly_v20260430.csv"),
]
if input_root.exists():
    for pattern in ["*/input/submission_032903_base.csv", "*/submission_032903_base.csv", "*/*/submission_032684_lynx_aliked_prob098_mergeonly_v20260430.csv", "*/*/*/submission_032684_lynx_aliked_prob098_mergeonly_v20260430.csv"]:
        base_submissions.extend(sorted(input_root.glob(pattern)))

base_submission_path = first_existing(base_submissions)

In [ ]:
metadata = pd.read_csv(data_root / "metadata.csv").copy()
if "dataset" not in metadata.columns:
    metadata["dataset"] = metadata["path"].astype(str).str.replace("\\", "/", regex=False).str.split("/").str[1]
if "split" not in metadata.columns:
    paths = metadata["path"].astype(str).str.replace("\\", "/", regex=False)
    metadata["split"] = np.where(paths.str.contains("/test/"), "test", "train")
metadata["image_id"] = metadata["image_id"].astype(int)

test_rows = metadata[metadata["split"].eq("test")].copy()
sample = pd.read_csv(data_root / "sample_submission.csv")[["image_id"]].copy()
sample["image_id"] = sample["image_id"].astype(int)
base = pd.read_csv(base_submission_path).copy()
base["image_id"] = base["image_id"].astype(int)

In [ ]:
class UnionFind:
    def __init__(self, ids):
        self.parent = {int(image_id): int(image_id) for image_id in ids}
        self.size = {int(image_id): 1 for image_id in ids}

    def find(self, image_id):
        image_id = int(image_id)
        while self.parent[image_id] != image_id:
            self.parent[image_id] = self.parent[self.parent[image_id]]
            image_id = self.parent[image_id]
        return image_id

    def union(self, a, b):
        root_a = self.find(a)
        root_b = self.find(b)
        if root_a == root_b:
            return False
        if self.size[root_a] < self.size[root_b]:
            root_a, root_b = root_b, root_a
        self.parent[root_b] = root_a
        self.size[root_a] += self.size[root_b]
        return True


def compact_species_labels(submission, sample_submission, test_metadata):
    parts = []
    sample_ids = set(sample_submission["image_id"].astype(int))
    scoped_metadata = test_metadata[test_metadata["image_id"].isin(sample_ids)]
    for species, species_rows in scoped_metadata.groupby("dataset", sort=True):
        ids = set(species_rows["image_id"].astype(int))
        species_submission = submission[submission["image_id"].astype(int).isin(ids)].copy()
        groups = []
        for _, members in species_submission.groupby("cluster"):
            groups.append(sorted(members["image_id"].astype(int).tolist()))
        labels = {}
        for idx, members in enumerate(sorted(groups, key=lambda values: (min(values), len(values), max(values)))):
            for image_id in members:
                labels[image_id] = f"cluster_{species}_{idx}"
        parts.append(pd.DataFrame({"image_id": sorted(ids), "cluster": [labels[image_id] for image_id in sorted(ids)]}))
    final = pd.concat(parts, ignore_index=True)
    final = sample_submission[["image_id"]].merge(final, on="image_id", how="left")
    return final[["image_id", "cluster"]]

In [ ]:
salamander_ids = test_rows[test_rows["dataset"].eq(SPECIES)]["image_id"].astype(int).tolist()
salamander_base = base[base["image_id"].isin(salamander_ids)].copy()
uf = UnionFind(salamander_ids)

for _, group in salamander_base.groupby("cluster"):
    image_ids = group["image_id"].astype(int).tolist()
    for a, b in itertools.combinations(image_ids, 2):
        uf.union(a, b)

for a, b in TRANSITIONS:
    uf.union(a, b)

groups = {}
for image_id in salamander_ids:
    groups.setdefault(uf.find(image_id), []).append(image_id)

labels = {}
for idx, members in enumerate(sorted(groups.values(), key=lambda values: (min(values), len(values), max(values)))):
    for image_id in members:
        labels[image_id] = f"cluster_{SPECIES}_{idx}"

submission = base.copy()
mask = submission["image_id"].isin(labels)
submission.loc[mask, "cluster"] = submission.loc[mask, "image_id"].map(labels)
submission = compact_species_labels(submission, sample, test_rows)

In [ ]:
kaggle_output = Path("/kaggle/working")
output_path = kaggle_output / "submission.csv" if kaggle_output.exists() and str(data_root).replace("\\", "/").startswith("/kaggle/") else work_dir / "submission.csv"
submission.to_csv(output_path, index=False, lineterminator="\n")
actual_sha256 = hashlib.sha256(output_path.read_bytes()).hexdigest()
shape = submission.merge(test_rows[["image_id", "dataset"]], on="image_id", how="left").groupby("dataset", sort=True).agg(n_images=("image_id", "size"), n_clusters=("cluster", "nunique"), max_cluster_size=("cluster", lambda values: values.value_counts().max()), singletons=("cluster", lambda values: int((values.value_counts() == 1).sum()))).reset_index()
print(output_path)
print(len(submission))
print(actual_sha256)
print(actual_sha256 == EXPECTED_SHA256)
print(shape.to_string(index=False))